# Visualization 3: Gender Effects on Name Popularity

**Design**: Dual area chart M/F — two superposed area charts for a given mixed-gender name.
The crossing zone indicates a popularity inversion between boys and girls.

- X-axis: year
- Y-axis: number of births
- Two areas: one for girls (F), one for boys (M)
- Interactive: dropdown to select a mixed-gender name

In [ ]:
import pandas as pd
import altair as alt

## Load and prepare data

In [ ]:
df = pd.read_csv('dpt2020.csv', sep=';', dtype={'annais': str, 'dpt': str})

# Drop rare names and unknown years
df = df[df['preusuel'] != '_PRENOMS_RARES']
df = df[df['annais'] != 'XXXX']
df['annais'] = df['annais'].astype(int)

# Aggregate nationally (sum over all departments)
national = df.groupby(['preusuel', 'sexe', 'annais'], as_index=False)['nombre'].sum()

print(f'{len(national)} rows after aggregation')
national.head()

## Find mixed-gender names (given to both sexes)

In [ ]:
# Names that appear for both sexe=1 (M) and sexe=2 (F)
sexes_per_name = national.groupby('preusuel')['sexe'].nunique()
mixed_names = sexes_per_name[sexes_per_name == 2].index.tolist()

# Keep only mixed names, require a minimum total count to avoid noise
mixed_df = national[national['preusuel'].isin(mixed_names)]
total_per_name = mixed_df.groupby('preusuel')['nombre'].sum()
popular_mixed = total_per_name[total_per_name >= 5000].index.tolist()
popular_mixed.sort()

print(f'{len(popular_mixed)} popular mixed-gender names')
print(popular_mixed[:20])

## Build the dual area chart

In [ ]:
# Prepare data: only mixed names, label sexe
viz_df = mixed_df[mixed_df['preusuel'].isin(popular_mixed)].copy()
viz_df['gender'] = viz_df['sexe'].map({1: 'Garçons', 2: 'Filles'})

# Dropdown selector
name_selector = alt.binding_select(options=popular_mixed, name='Prénom : ')
name_param = alt.param(name='selected_name', value='CAMILLE', bind=name_selector)

# Color scale: blue for boys, pink for girls
color_scale = alt.Scale(
    domain=['Garçons', 'Filles'],
    range=['#4C9BE8', '#E8748C']
)

area = (
    alt.Chart(viz_df)
    .mark_area(opacity=0.55, interpolate='monotone')
    .encode(
        x=alt.X('annais:Q', title='Année', axis=alt.Axis(format='d')),
        y=alt.Y('nombre:Q', title='Nombre de naissances'),
        color=alt.Color('gender:N', scale=color_scale, title='Sexe'),
        tooltip=[
            alt.Tooltip('annais:Q', title='Année'),
            alt.Tooltip('gender:N', title='Sexe'),
            alt.Tooltip('nombre:Q', title='Naissances', format=','),
        ]
    )
    .transform_filter(alt.datum.preusuel == name_param)
    .add_params(name_param)
    .properties(
        width=750,
        height=380,
        title=alt.TitleParams(
            text='Popularité M/F d\'un prénom mixte au fil du temps',
            subtitle='La zone de croisement indique une inversion de popularité entre les sexes',
            fontSize=16,
            subtitleFontSize=12,
        )
    )
)

# Vertical rule to highlight crossover points
# We compute crossover years: where the dominant sex flips
area

## Enhanced version: add a crossover indicator line

In [ ]:
# Pivot to compute difference (Filles - Garçons) to find crossover zones
pivot = viz_df.pivot_table(
    index=['preusuel', 'annais'], columns='gender', values='nombre', fill_value=0
).reset_index()
pivot.columns.name = None

# diff > 0 means Filles dominant, < 0 means Garçons dominant
pivot['diff'] = pivot.get('Filles', 0) - pivot.get('Garçons', 0)

diff_line = (
    alt.Chart(pivot)
    .mark_line(color='black', strokeDash=[4, 4], strokeWidth=1.2, opacity=0.4)
    .encode(
        x=alt.X('annais:Q', title='Année', axis=alt.Axis(format='d')),
        y=alt.Y('diff:Q', title='Filles − Garçons'),
        tooltip=[
            alt.Tooltip('annais:Q', title='Année'),
            alt.Tooltip('diff:Q', title='Filles − Garçons', format=','),
        ]
    )
    .transform_filter(alt.datum.preusuel == name_param)
    .add_params(name_param)
    .properties(width=750, height=120, title='Différence Filles − Garçons (0 = parité)')
)

zero_line = (
    alt.Chart(pd.DataFrame({'y': [0]}))
    .mark_rule(color='red', strokeDash=[3, 3], opacity=0.6)
    .encode(y='y:Q')
)

final_chart = alt.vconcat(
    area,
    diff_line + zero_line
).configure_legend(
    orient='top-right',
    labelFontSize=13,
    titleFontSize=13,
).configure_axis(
    labelFontSize=12,
    titleFontSize=13,
)

final_chart

## Save to HTML

In [ ]:
final_chart.save('viz3_gender.html')
print('Saved to viz3_gender.html')